## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# (local) No hace falta montar Google Drive: los datasets estan en datasets/


Para correr la siguiente celda es fundamental, POR UNICA VEZ, seguir los siguientes pasos

* Registrar usuario en Kaggle con la cuenta de email de la Universidad Austral
* Hacer el "Join Competition"  a la competencia de  Labo 3
* Generar el archivo kaggle.json  a partir de   https://www.kaggle.com/settings/account  y presione  "Create Legacy API Key"
* Crear carpeta  labo3  en  el Google Drive
* Dentro de la carpeta labo3 crear carpeta   kaggle
* Subir a la carpeta kaggle el archivo  kaggle.json


In [2]:
# (local) Datasets ya descargados en datasets/ — no hace falta wget


# 1 Exploratory Data Analysis

In [3]:
import os as os
import duckdb
import plotly.express as px

In [4]:
# defino los parametros
PARAM = {'experimento':'EDA-101',
  'kaggle_competition':'labo-iii-2026-ba'
}

In [5]:
# creo la carpeta del experimento
ruta = "./exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
pass  # (local) sin chdir

./exp/EDA-101


## 1.1 Creacion de tablas

Para el Exploratory Data Analysis utilizo DuckDB
<br>Cargo los archivos a utilizar en TABLAS de DuckDB

In [6]:
con = duckdb.connect()

In [7]:
# creo la tabla del sell-in  transformando el campo periodo a tipo DATE

con.execute("""
    CREATE OR REPLACE TABLE tb_sellin AS
    SELECT CAST(strptime(CAST( periodo AS VARCHAR), '%Y%m') AS DATE) periodo,
           customer_id,
           product_id,
           plan_precios_cuidados,
           cust_request_qty,
           cust_request_tn,
           tn
    FROM read_csv_auto('/Users/giulialvaro/Library/CloudStorage/OneDrive-Personal/estudios/Austral/Materias/018-Laboratorio de Implementación III/labo3-2026ba/datasets/sell-in.txt.gz')
    ORDER BY product_id, periodo, customer_id
""")

In [8]:
# creo la tabla que posee la definicion de los productos

con.execute("""
    CREATE OR REPLACE TABLE tb_productos AS
    SELECT *
    FROM read_csv_auto('/Users/giulialvaro/Library/CloudStorage/OneDrive-Personal/estudios/Austral/Materias/018-Laboratorio de Implementación III/labo3-2026ba/datasets/tb_productos.txt')
    ORDER BY product_id
""")

## 1.2 EDA Ventas Totales

In [9]:
# creo las ventas por producto

con.execute("""
    CREATE OR REPLACE TABLE tb_ventas_global AS
    SELECT periodo,
           SUM(tn) tn
    FROM  tb_sellin
    GROUP BY periodo
    ORDER BY periodo
""")



In [10]:
con.sql("""
SELECT  year(periodo) yean,
        round( SUM(tn) ) tn
FROM    tb_ventas_global
GROUP BY year(periodo)
ORDER BY year(periodo)
""").show()

┌───────┬──────────┐
│ yean  │    tn    │
│ int64 │  double  │
├───────┼──────────┤
│  2017 │ 500310.0 │
│  2018 │ 434448.0 │
│  2019 │ 390230.0 │
└───────┴──────────┘



¿Cómo interpreta lo que viene sucediendo año a año con las ventas?

In [11]:
# Grafico global de ventas
gra = px.line(
    con.sql("SELECT * FROM tb_ventas_global").df(),
    x="periodo",
    y="tn",
    title="Ventas Totales",
    labels={"periodo": "Periodo", "tn": "Toneladas"}
)

# Display the plot
gra.update_yaxes(rangemode="tozero")
gra.show()

In [12]:
# Grafico global de ventas con tendencia
gra = px.scatter(
    con.sql("SELECT * FROM tb_ventas_global").df(),
    x="periodo",
    y="tn",
    title="Ventas Totales con tendencia",
    labels={"periodo": "Periodo", "tn": "Toneladas"},
    trendline='ols',
    trendline_color_override='red'
)

# Display the plot
gra.update_traces(mode='lines')
gra.update_yaxes(rangemode="tozero")
gra.show()

¿Cómo es la tendencia de las ventas en los últimos 3 años?

## 1.3  EDA Clientes
¿Cuánto market share tiene los clientes?

In [13]:
# Creacion de tabla de clientes de 2019
con.execute("""
CREATE OR REPLACE TABLE tb_clientes_2019 AS
SELECT  customer_id,
        SUM(tn) tn,
        (SUM(tn) * 100.0) / SUM(SUM(tn)) OVER () tn_pct
FROM    tb_sellin
WHERE   periodo between '2019-01-01' AND '2019-12-01'
GROUP BY customer_id
ORDER BY tn DESC
""")

In [14]:
gra = px.bar(
    con.sql("SELECT tn_pct FROM tb_clientes_2019").df(),
    y="tn_pct",
    title="Densidad de tn por cliente",
    labels={ "tn_pct": "Toneladas Porcentaje"}
)

# Display the plot
gra.update_yaxes(rangemode="tozero")
gra.show()

In [15]:
# Cantidad de clientes distintos en 2019

con.sql("""
SELECT  COUNT( DISTINCT customer_id )
FROM    tb_clientes_2019
""")

┌─────────────────────────────┐
│ count(DISTINCT customer_id) │
│            int64            │
├─────────────────────────────┤
│                         534 │
└─────────────────────────────┘

In [16]:
# el market share acumulado de los mejores clientes
con.sql("""
SELECT  row_number() OVER(),
        customer_id,
        tn,
        tn_pct,
        SUM(tn_pct) OVER (ORDER BY tn_pct DESC) AS tn_pct_acumulado
FROM    tb_clientes_2019
ORDER BY tn DESC
LIMIT 20
""").show()

┌──────────────────────┬─────────────┬────────────────────┬────────────────────┬────────────────────┐
│ row_number() OVER () │ customer_id │         tn         │       tn_pct       │  tn_pct_acumulado  │
│        int64         │    int64    │       double       │       double       │       double       │
├──────────────────────┼─────────────┼────────────────────┼────────────────────┼────────────────────┤
│                    1 │       10001 │ 33685.890010000025 │   8.63232198959663 │   8.63232198959663 │
│                    2 │       10002 │  25948.00374999998 │  6.649416811928267 │ 15.281738801524897 │
│                    3 │       10003 │        20514.40695 │  5.257007196943548 │ 20.538745998468446 │
│                    4 │       10004 │ 15890.077299999999 │ 4.0719798008145345 │  24.61072579928298 │
│                    5 │       10005 │ 14958.134389999992 │   3.83316077977476 │ 28.443886579057743 │
│                    6 │       10006 │ 14147.636789999995 │ 3.6254632466854373 │  

Los primeros 13 productos representan el 50% de las ventas !

In [17]:
# el market share acumulado de los PEORES clientes
con.sql("""
SELECT  row_number() OVER(),
        customer_id,
        tn,
        tn_pct,
        SUM(tn_pct) OVER (ORDER BY tn_pct DESC) AS tn_pct_acumulado
FROM    tb_clientes_2019
ORDER BY tn ASC
LIMIT 300
""").show()

┌──────────────────────┬─────────────┬─────────────────────┬────────────────────────┬───────────────────┐
│ row_number() OVER () │ customer_id │         tn          │         tn_pct         │ tn_pct_acumulado  │
│        int64         │    int64    │       double        │         double         │      double       │
├──────────────────────┼─────────────┼─────────────────────┼────────────────────────┼───────────────────┤
│                  534 │       10604 │ 0.19666000000000003 │  5.039595041039772e-05 │ 99.99999999999983 │
│                  533 │       10524 │             0.22126 │  5.669992874913352e-05 │ 99.99994960404942 │
│                  532 │       10618 │             0.42129 │ 0.00010795947294008163 │ 99.99989290412067 │
│                  531 │       10574 │ 0.44403000000000004 │ 0.00011378680901418133 │ 99.99978494464773 │
│                  530 │       10581 │  0.5010100000000001 │ 0.00012838846290609866 │ 99.99967115783872 │
│                  529 │       10593 │  0.6977

Los 300 clientes menos importantes representan MENOS del 3% de las toneladas vendidas

In [18]:
# evolucion de algunos clientes

clientes=[10001, 10002, 10012]

gra=px.line(
    con.sql(f"SELECT customer_id, periodo, SUM(tn) tn FROM tb_sellin WHERE customer_id in {clientes} GROUP BY customer_id, periodo ORDER BY 1,2").df(),
    x="periodo",
    y="tn",
    color='customer_id',
    title='customer_id =' + str(clientes),
    labels={"periodo": "Periodo", "tn": "Toneladas"}
  )

# Display the plot
gra.update_traces(mode='lines')
gra.update_yaxes(rangemode="tozero")
gra.show()

## 1.4 EDA productos individuales
Market shared de los productos

In [19]:
con.execute("""
CREATE OR REPLACE TABLE tb_productos_201912 AS
SELECT  product_id,
        SUM(tn) tn,
        (SUM(tn) * 100.0) / SUM(SUM(tn)) OVER () tn_pct
FROM    tb_sellin
WHERE   periodo between '2019-01-01' and '2019-12-01'
GROUP BY product_id
ORDER BY tn DESC
""")

In [20]:
con.sql("""
SELECT *
FROM   tb_productos_201912
""").show()

┌────────────┬──────────────────────┬────────────────────────┐
│ product_id │          tn          │         tn_pct         │
│   int64    │        double        │         double         │
├────────────┼──────────────────────┼────────────────────────┤
│      20001 │    17456.79264000003 │      4.473465149039151 │
│      20002 │   14105.245700000058 │     3.6146001363962186 │
│      20003 │    9419.716889999925 │      2.413889887462744 │
│      20005 │    8019.241249999978 │     2.0550050054104325 │
│      20004 │    7526.583939999965 │     1.9287569968470204 │
│      20009 │    6495.871040000001 │     1.6646272490805425 │
│      20032 │    6493.670259999988 │     1.6640632787777618 │
│      20006 │    5743.364500000031 │     1.4717904633928656 │
│      20007 │     5209.65367000002 │      1.335022109268118 │
│      20010 │    5154.895930000037 │     1.3209899301283614 │
│        ·   │            ·         │              ·         │
│        ·   │            ·         │              ·   

In [21]:
con.sql("""
SELECT  product_id,
        tn,
        tn_pct,
        SUM(tn_pct) OVER (ORDER BY tn_pct DESC) AS tn_pct_acumulado
FROM    tb_productos_201912
ORDER BY tn DESC
""").show()

┌────────────┬──────────────────────┬────────────────────────┬────────────────────┐
│ product_id │          tn          │         tn_pct         │  tn_pct_acumulado  │
│   int64    │        double        │         double         │       double       │
├────────────┼──────────────────────┼────────────────────────┼────────────────────┤
│      20001 │    17456.79264000003 │      4.473465149039151 │  4.473465149039151 │
│      20002 │   14105.245700000058 │     3.6146001363962186 │   8.08806528543537 │
│      20003 │    9419.716889999925 │      2.413889887462744 │ 10.501955172898114 │
│      20005 │    8019.241249999978 │     2.0550050054104325 │ 12.556960178308547 │
│      20004 │    7526.583939999965 │     1.9287569968470204 │ 14.485717175155568 │
│      20009 │    6495.871040000001 │     1.6646272490805425 │  16.15034442423611 │
│      20032 │    6493.670259999988 │     1.6640632787777618 │  17.81440770301387 │
│      20006 │    5743.364500000031 │     1.4717904633928656 │ 19.2861981664

In [22]:
def graficar_un_producto(producto):
  gra = px.scatter(
      con.sql(f"SELECT periodo, SUM(tn) tn FROM tb_sellin WHERE product_id={producto} GROUP BY periodo").df(),
      x="periodo",
      y="tn",
      title=con.sql(f"SELECT CONCAT_WS(', ', *COLUMNS('.*')) FROM tb_productos WHERE product_id={producto}").fetchone()[0],
      labels={"periodo": "Periodo", "tn": "Toneladas"}
  )

  # Display the plot
  gra.update_traces(mode='lines')
  gra.update_yaxes(rangemode="tozero")
  gra.update_xaxes(range=['2017-01-01', '2019-12-01'])
  gra.show()

In [23]:
def graficar_productos(productos):
    for prod in productos:
      graficar_un_producto(prod)

In [24]:
# gnero graficos INDEPENDIENTES de productos

graficar_productos( [20001, 20002, 20003, 20004])

## 1.5  EDA multiples productos

In [25]:
productos = [ 20001, 20002]

tbl=con.sql(f"SELECT product_id, periodo, SUM(tn) tn FROM tb_sellin WHERE product_id in {productos} GROUP BY product_id, periodo ORDER BY 1, 2").df()
display(tbl)

,product_id,periodo,tn
0,20001,2017-01-01,934.77222
1,20001,2017-02-01,798.01620
2,20001,2017-03-01,1303.35771
3,20001,2017-04-01,1069.96130
4,20001,2017-05-01,1502.20132
...,...,...,...
67,20002,2019-08-01,813.78215
68,20002,2019-09-01,1090.18771
69,20002,2019-10-01,1979.53635
70,20002,2019-11-01,1423.57739


In [26]:
def graficar_multiples_productos(productos):
  gra = px.line(
      con.sql(f"SELECT product_id, periodo, SUM(tn) tn FROM tb_sellin WHERE product_id in {productos} GROUP BY product_id, periodo ORDER BY 1,2").df(),
      x="periodo",
      y="tn",
      color="product_id",
      title="Multiples Productos",
      labels={"periodo": "Periodo", "tn": "Toneladas"}
  )

  # Display the plot
  gra.update_yaxes(rangemode="tozero")
  gra.update_xaxes(range=['2017-01-01', '2019-12-01'])
  gra.show()

In [27]:
def graficar_union_productos(productos):
  gra = px.line(
      con.sql(f"SELECT periodo, SUM(tn) tn FROM tb_sellin WHERE product_id in {productos} GROUP BY periodo ORDER BY 1").df(),
      x="periodo",
      y="tn",
      title="Productos in " + str(productos),
      labels={"periodo": "Periodo", "tn": "Toneladas"}
  )

  # Display the plot
  gra.update_yaxes(rangemode="tozero")
  gra.update_xaxes(range=['2017-01-01', '2019-12-01'])
  gra.show()

In [28]:
graficar_multiples_productos( [20001, 20002, 20003] )

In [29]:
graficar_union_productos( [20001, 20002, 20003] )

## 1.6  EDA Estacionalidad Mayonesas

In [30]:
con.sql("SELECT * FROM tb_productos WHERE cat3='Mayonesa'")

┌─────────┬──────────┬──────────┬─────────┬──────────┬────────────┬──────────────────────┐
│  cat1   │   cat2   │   cat3   │  brand  │ sku_size │ product_id │     descripcion      │
│ varchar │ varchar  │ varchar  │ varchar │  int64   │   int64    │       varchar        │
├─────────┼──────────┼──────────┼─────────┼──────────┼────────────┼──────────────────────┤
│ FOODS   │ ADEREZOS │ Mayonesa │ NATURA  │      475 │      20003 │ Regular sin TACC     │
│ FOODS   │ ADEREZOS │ Mayonesa │ NATURA  │      240 │      20004 │ Regular sin TACC     │
│ FOODS   │ ADEREZOS │ Mayonesa │ NATURA  │      120 │      20005 │ Regular sin TACC     │
│ FOODS   │ ADEREZOS │ Mayonesa │ NATURA  │      950 │      20019 │ Regular sin TACC     │
│ FOODS   │ ADEREZOS │ Mayonesa │ NATURA  │      475 │      20046 │ Light sin TACC       │
│ FOODS   │ ADEREZOS │ Mayonesa │ MAYOS3  │      475 │      20084 │ Reguar sin TACC      │
│ FOODS   │ ADEREZOS │ Mayonesa │ MAJESTA │      475 │      20107 │ Mayonesa Tradicional │

In [31]:
algunas_mayonesas = [20003, 20004, 20005]

In [32]:
con.sql(f"SELECT * FROM tb_productos WHERE product_id in {productos}")

┌─────────┬─────────────┬─────────┬─────────┬──────────┬────────────┬────────────────────┐
│  cat1   │    cat2     │  cat3   │  brand  │ sku_size │ product_id │    descripcion     │
│ varchar │   varchar   │ varchar │ varchar │  int64   │   int64    │      varchar       │
├─────────┼─────────────┼─────────┼─────────┼──────────┼────────────┼────────────────────┤
│ HC      │ ROPA LAVADO │ Liquido │ ARIEL   │     3000 │      20001 │ genoma             │
│ HC      │ ROPA LAVADO │ Liquido │ LIMPIEX │     3000 │      20002 │ Maquina 1er lavado │
└─────────┴─────────────┴─────────┴─────────┴──────────┴────────────┴────────────────────┘

In [33]:
graficar_multiples_productos(algunas_mayonesas)

In [34]:
graficar_union_productos(algunas_mayonesas)

¿Qué estacionalidad se observa para las mayonesas?

## 1.7 EDA Estacionalidad Sopas
Dado que entre que usualmente hay DOS MESES entre que el caminón sale por el portón de la planta de La Multinacional y es escaneado en la linea de caja del supermercado
<br> ¿En qué mes las familias empiezan a compar mas sopas?
<br> ¿En qué mes la multinacional vende más sopas?

In [35]:
con.sql("SELECT * FROM tb_productos WHERE cat3='Sopas'")

┌─────────┬────────────────┬─────────┬─────────┬──────────┬────────────┬─────────────────────────┐
│  cat1   │      cat2      │  cat3   │  brand  │ sku_size │ product_id │       descripcion       │
│ varchar │    varchar     │ varchar │ varchar │  int64   │   int64    │         varchar         │
├─────────┼────────────────┼─────────┼─────────┼──────────┼────────────┼─────────────────────────┤
│ FOODS   │ SOPAS Y CALDOS │ Sopas   │ MAGGI   │       10 │      20234 │ Sopa 21                 │
│ FOODS   │ SOPAS Y CALDOS │ Sopas   │ MAGGI   │       10 │      20265 │ Sopa vegetales          │
│ FOODS   │ SOPAS Y CALDOS │ Sopas   │ MAGGI   │       10 │      20302 │ Sopa fideos y vegetales │
│ FOODS   │ SOPAS Y CALDOS │ Sopas   │ MAGGI   │       10 │      20305 │ Sopa fideos             │
│ FOODS   │ SOPAS Y CALDOS │ Sopas   │ MAGGI   │       10 │      20350 │ Sopla 22                │
│ FOODS   │ SOPAS Y CALDOS │ Sopas   │ MAGGI   │       10 │      20398 │ Sopa 1                  │
│ FOODS   

In [36]:
algunas_sopas=[20234, 20265, 20302]

In [37]:
graficar_multiples_productos(algunas_sopas)

In [38]:
graficar_union_productos(algunas_sopas)

## 1.8 EDA Productos Infantiles
¿Qué forma tiene el sell-out de los productos nuevos?
<br>¿Los productos nuevos, canibalizan a otros productos de la misma familia de productos de La Multinacional?

In [39]:
mostazas_infantes=[21144, 21146, 21154]

In [40]:
graficar_multiples_productos(mostazas_infantes)

In [41]:
mostazas_varias=[21144, 21146, 21154, 20884]

In [42]:
con.sql(f"SELECT * FROM tb_productos WHERE product_id in {mostazas_varias}")

┌─────────┬──────────┬─────────┬──────────┬──────────┬────────────┬──────────────────────────┐
│  cat1   │   cat2   │  cat3   │  brand   │ sku_size │ product_id │       descripcion        │
│ varchar │ varchar  │ varchar │ varchar  │  int64   │   int64    │         varchar          │
├─────────┼──────────┼─────────┼──────────┼──────────┼────────────┼──────────────────────────┤
│ FOODS   │ ADEREZOS │ Mostaza │ MOSTAZA1 │      275 │      20884 │ abejas                   │
│ FOODS   │ ADEREZOS │ Mostaza │ MOSTAZA1 │      180 │      21144 │ Pimienta                 │
│ FOODS   │ ADEREZOS │ Mostaza │ MOSTAZA1 │      180 │      21146 │ colchon de finas hierbas │
│ FOODS   │ ADEREZOS │ Mostaza │ MOSTAZA1 │      180 │      21154 │ Mostaza Ahumada          │
└─────────┴──────────┴─────────┴──────────┴──────────┴────────────┴──────────────────────────┘

In [43]:
graficar_multiples_productos(mostazas_varias)

Podria pasar que para 2019-09 el product_id=20884 se vendió menos porque fue canibalizado por los nuevos tres productos?

## 1.9 EDA Productos discontinuados

¿Los productos discontinuados son reemplazados por otros que toman su lugar?

In [44]:
mayonesas_discontinuadas=[20494, 20554]

In [45]:
graficar_multiples_productos(mayonesas_discontinuadas)

In [46]:
mayonesas=[20494, 20554, 20580]

In [47]:
graficar_multiples_productos(mayonesas)

## Hallazgo propio 1 — ¿Cuántos de los 780 productos a predecir tienen historia completa?

In [48]:
# cargo la lista de los 780 productos que hay que predecir
con.execute("""
    CREATE OR REPLACE TABLE tb_apredecir AS
    SELECT * FROM read_csv_auto('/Users/giulialvaro/Library/CloudStorage/OneDrive-Personal/estudios/Austral/Materias/018-Laboratorio de Implementación III/labo3-2026ba/datasets/product_id_apredecir201912.txt')
""")

con.sql("""
    WITH meses AS (
        SELECT product_id, COUNT(DISTINCT periodo) meses_2019
        FROM tb_sellin
        WHERE periodo BETWEEN '2019-01-01' AND '2019-12-01'
        GROUP BY product_id)
    SELECT CASE WHEN m.meses_2019 = 12 THEN '12 meses (completo)'
                WHEN m.meses_2019 IS NULL THEN 'sin ventas 2019'
                ELSE '<12 meses (hueco/nuevo)' END grupo,
           COUNT(*) cant
    FROM tb_apredecir a
    LEFT JOIN meses m ON a.product_id = m.product_id
    GROUP BY 1 ORDER BY 2 DESC
""").show()

┌─────────────────────────┬───────┐
│          grupo          │ cant  │
│         varchar         │ int64 │
├─────────────────────────┼───────┤
│ 12 meses (completo)     │   650 │
│ <12 meses (hueco/nuevo) │   130 │
└─────────────────────────┴───────┘



## Hallazgo propio 2 — Concentración: pocos productos = casi toda la venta (Pareto)

In [49]:
con.sql("""
    WITH p AS (
        SELECT product_id, SUM(tn) tn
        FROM tb_sellin
        WHERE periodo BETWEEN '2019-01-01' AND '2019-12-01'
        GROUP BY product_id),
    acum AS (
        SELECT product_id,
               100.0*SUM(tn) OVER (ORDER BY tn DESC)/SUM(tn) OVER () pct_acum,
               COUNT(*) OVER () total
        FROM p)
    SELECT v.umbral||'% de las tn' cubre,
           COUNT(*)+1 productos,
           ROUND(100.0*(COUNT(*)+1)/MAX(acum.total),1)||'% del catalogo' representan
    FROM acum CROSS JOIN (VALUES (50),(80),(90)) v(umbral)
    WHERE acum.pct_acum <= v.umbral
    GROUP BY v.umbral ORDER BY v.umbral
""").show()

┌───────────────┬───────────┬────────────────────┐
│     cubre     │ productos │    representan     │
│    varchar    │   int64   │      varchar       │
├───────────────┼───────────┼────────────────────┤
│ 50% de las tn │        50 │ 4.6% del catalogo  │
│ 80% de las tn │       177 │ 16.3% del catalogo │
│ 90% de las tn │       314 │ 29.0% del catalogo │
└───────────────┴───────────┴────────────────────┘



## Hallazgo propio 3 — La distribución de ventas es muy asimétrica (cola larga)

In [51]:
con.sql("""
    WITH pm AS (
        SELECT product_id, periodo, SUM(tn) tn
        FROM tb_sellin GROUP BY product_id, periodo)
    SELECT ROUND(AVG(tn),1) media, ROUND(MEDIAN(tn),1) mediana,
           ROUND(AVG(tn)/MEDIAN(tn),1) ratio, ROUND(MAX(tn),0) maximo
    FROM pm
""").show()

┌────────┬─────────┬────────┬────────┐
│ media  │ mediana │ ratio  │ maximo │
│ double │ double  │ double │ double │
├────────┼─────────┼────────┼────────┤
│   42.4 │     9.8 │    4.3 │ 2295.0 │
└────────┴─────────┴────────┴────────┘



In [52]:
import plotly.express as px
pm = con.sql("SELECT SUM(tn) tn FROM tb_sellin GROUP BY product_id, periodo").df()
px.histogram(pm, x="tn", nbins=100, title="Distribución de tn (producto×mes) — cola larga").show()